**SAPIANS** · SCC5819 · MÓDULO 10 — PFI · MOLNAR, CAP. 23

# PENDENTE — auditoria do agregado adicional de PFI

O [walkthrough deste roteiro](srag_course_pfi_walkthrough.ipynb) prepara a
leitura de um agregado adicional. A seleção no treino completo foi
interrompida antes da etapa final; o agregado ainda não existe. A auditoria
vigente da aula está no [internals principal](../pfi_internals.ipynb).

[README do módulo](../../README.md) · [outline da aula](../../lecture/outline.md) ·
[Colab](https://colab.research.google.com/github/wbendinelli/interpretable-ml-lectures/blob/main/modules/10-pfi/notebooks/archive/srag_course_pfi_internals.ipynb)

---

**Objetivos.** Preparar conferências de integridade, cardinalidade, diferença,
razão, SD, quantis e rankings independentes do resumidor do executor.
Distinguir checagens matemáticas do agregado de uma reprodução dos erros
a partir das notificações, sem ajustar outro modelo.

**Estado da evidência: PENDENTE.** A execução atual imprime a ausência do
agregado final; nenhuma auditoria empírica adicional foi concluída. A
eventual retomada do estudo depende de nova decisão de escopo.


In [1]:
%pip install -q numpy pandas scipy matplotlib


Note: you may need to restart the kernel to use updated packages.


In [2]:
import hashlib
import json
import os
import pathlib
import subprocess
import sys
import tempfile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

RANDOM_STATE = 42
REPO = pathlib.Path(
    os.environ.get("INTERPRETABLE_ML_REPO", pathlib.Path.cwd())
).resolve()
while REPO != REPO.parent and not (REPO / "tools/sapians.py").exists():
    REPO = REPO.parent
if not (REPO / "tools/sapians.py").exists():
    REPO = pathlib.Path(tempfile.gettempdir()) / "interpretable-ml-lectures"
    if not REPO.exists():
        subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "https://github.com/wbendinelli/interpretable-ml-lectures.git",
                str(REPO),
            ],
            check=True,
            capture_output=True,
        )
sys.path.insert(0, str(REPO / "tools"))
import sapians as SP

work_env = os.environ.get("INTERPRETABLE_ML_WORK")
WORK = (
    pathlib.Path(work_env).resolve()
    if work_env
    else pathlib.Path(tempfile.mkdtemp(prefix="course-pfi-pending-"))
)
WORK.mkdir(parents=True, exist_ok=True)
assert not WORK.is_relative_to(REPO)
os.chdir(WORK)
SP.aplicar(tema="global")
print("Perfil visual: Global; fonte ativa:", SP.FONTE_ATUAL)
RECORD = REPO / "modules/00-dataset/gold/course_pfi_v2.json"
RESULT = json.loads(RECORD.read_text()) if RECORD.exists() else None
READY = RESULT is not None and RESULT.get("status") == "complete"
if (
    RESULT is not None
    and not READY
    and RESULT.get("status") not in {"pending", "incomplete"}
):
    raise ValueError("Estado não reconhecido no registro público da PFI.")
if READY:
    protocol = RESULT["protocol"]
    original = float(RESULT["original"]["logloss"])
    rows = RESULT["rows"]
    assert protocol["version"] == "course-full-logloss-pfi-v2"
    assert original > 0 and np.isfinite(original)
else:
    protocol, original, rows = None, None, []
print("Fonte esperada: modules/00-dataset/gold/course_pfi_v2.json")
print(
    "Estado:",
    "CONCLUÍDO — resultados agregados disponíveis."
    if READY
    else "PENDENTE — estudo no treino completo interrompido antes da seleção final; agregado ausente.",
)
print(
    "Unidade prevista: notificação de hospitalização; este relatório não carrega dados individuais."
)
print("Nenhum ajuste, seleção de modelo ou cálculo de novas previsões é executado.")

from scipy.stats import rankdata


Perfil visual: Global; fonte ativa: Satoshi
Fonte esperada: modules/00-dataset/gold/course_pfi_v2.json
Estado: PENDENTE — estudo no treino completo interrompido antes da seleção final; agregado ausente.
Unidade prevista: notificação de hospitalização; este relatório não carrega dados individuais.
Nenhum ajuste, seleção de modelo ou cálculo de novas previsões é executado.


## §1 — Integridade do registro e identidade do preditor

Conferimos o hash do resultado e o de seu protocolo, além da ligação entre
seleção e artefato canônico. Esses hashes detectam divergências no documento;
não substituem acesso aos dados de origem nem uma reprodução do experimento.

Uma entrada deve aparecer uma única vez no ranking e ter um vetor completo
de perdas. O agregado é aceito como concluído somente quando seu estado é
explícito e seus contratos estão presentes.


In [3]:
def canonical_hash(value):
    encoded = json.dumps(
        value, sort_keys=True, ensure_ascii=False, allow_nan=False
    ).encode()
    return hashlib.sha256(encoded).hexdigest()


def verify_document(document, key):
    expected = document[key]
    assert isinstance(expected, str) and len(expected) == 64
    assert canonical_hash({k: v for k, v in document.items() if k != key}) == expected


if READY:
    verify_document(RESULT, "results_sha256")
    verify_document(protocol, "pfi_protocol_sha256")
    assert protocol["selection_sha256"] == protocol["model"]["selection_sha256"]
    assert protocol["canonical_model"] == protocol["model"]["canonical_model"]
    assert RESULT["integrity"]["fit_performed"] is False
    print("Hashes internos do resultado e do protocolo: conferem.")
    print("Versão da PFI:", protocol["version"])
    print("Versão do preditor:", protocol["model"]["version"])
    print("Família selecionada:", protocol["model"]["family"])
    print("Estratégia selecionada:", protocol["model"]["strategy"])
    print("Identificador do candidato:", protocol["model"]["candidate_id"])
    print("Regra de seleção:", protocol["model"]["selection_rule"])
    print("Ano da avaliação:", str(protocol["period"]))
    print("Registros de avaliação:", SP.pt_int(protocol["n_evaluation"]))
    print("Óbitos observados:", SP.pt_int(protocol["events"]))
    print("Entradas brutas:", SP.pt_int(len(protocol["features"])))
    print("Repetições por operação:", SP.pt_int(protocol["repeats"]))
    print("Semente de permutação:", SP.pt_int(protocol["permutation_seed"]))
    print(
        "Probabilidades recalibradas:",
        "sim" if protocol["model"]["probabilities_recalibrated"] else "não",
    )
    print(
        "Condição da avaliação: período temporal histórico já observado; análise exploratória."
    )
else:
    print(
        "PENDENTE — o registro adicional não contém preditor finalizado nem desempenho para relatar."
    )

if READY:
    feature_names = protocol["features"]
    assert len(feature_names) == 40 and len(set(feature_names)) == len(feature_names)
    assert [row["feature"] for row in rows] == feature_names
    assert set(RESULT["ranking"]) == set(feature_names)
    assert len(RESULT["ranking"]) == len(feature_names)
    assert protocol["split"] == "test" and protocol["period"] == 2024
    assert 0 < protocol["events"] < protocol["n_evaluation"]
    assert protocol["repeats"] >= 2
    assert protocol["positive_class"] == "death=1; survival=0"
    for row in rows:
        after = np.asarray(row["metrics"]["logloss"]["after"], dtype=float)
        assert after.shape == (protocol["repeats"],)
        assert np.isfinite(after).all() and (after >= 0).all()
    print("Contrato das entradas, estado e vetores completos: confere.")
    print("Vetores de perdas por entrada:", SP.pt_int(len(rows)))
else:
    print("PENDENTE — sem vetores finais, não há contrato empírico a aprovar.")


PENDENTE — o registro adicional não contém preditor finalizado nem desempenho para relatar.
PENDENTE — sem vetores finais, não há contrato empírico a aprovar.


**O que olhar.**

- Os identificadores ligam o agregado a uma seleção e a um modelo específicos.
- Uma lista incompleta não é promovida a ranking completo.
- A leitura do registro não chama o executor de PFI nem reconstrói o preditor.


## §2 — Recalcular diferença, razão e SD a partir das perdas registradas

Usamos diretamente as perdas após permutar, o erro original e as definições
matemáticas. O código abaixo não chama o resumidor do executor. Ele confronta
cada campo com um cálculo independente e recusa valores ausentes ou incompatíveis.

A verificação inclui a relação entre as dispersões das escalas. Um SD dos
sorteios e um erro padrão de Monte Carlo não se transformam, por esse teste,
em intervalo de confiança sobre notificações ou sobre uma população.


In [4]:
def audit_summary(original, measure, repeats):
    after = np.asarray(measure["after"], dtype=np.float64)
    difference = after-original
    ratio = after/original
    assert after.shape == (repeats,) and np.isfinite(after).all()
    np.testing.assert_allclose(measure["delta"], difference, rtol=0, atol=1e-14)
    np.testing.assert_allclose(measure["ratio"], ratio, rtol=0, atol=1e-14)
    expected = {
        "after_mean": np.mean(after), "after_sd": np.std(after, ddof=1),
        "mean": np.mean(difference), "sd": np.std(difference, ddof=1),
        "mcse": np.std(difference, ddof=1)/np.sqrt(repeats),
        "median": np.median(difference), "q25": np.quantile(difference, .25),
        "q75": np.quantile(difference, .75), "q05": np.quantile(difference, .05),
        "q95": np.quantile(difference, .95), "ratio_mean": np.mean(ratio),
        "ratio_sd": np.std(ratio, ddof=1),
        "relative_increase_mean": np.mean(ratio-1),
        "relative_increase_sd": np.std(ratio-1, ddof=1),
    }
    for key, value in expected.items():
        np.testing.assert_allclose(measure[key], value, rtol=0, atol=1e-12)
    np.testing.assert_allclose(measure["after_sd"], measure["sd"], rtol=0, atol=1e-14)
    np.testing.assert_allclose(measure["ratio_sd"], measure["sd"]/original, rtol=0, atol=1e-14)
    np.testing.assert_allclose(measure["ratio_mean"], 1+measure["mean"]/original, rtol=0, atol=1e-14)
    return max(abs(float(measure[key])-float(value)) for key, value in expected.items())

if READY:
    summary_audits = [{"entrada": row["label"], "maior desvio absoluto":
                      audit_summary(original, row["metrics"]["logloss"], protocol["repeats"])}
                     for row in rows]
    print(SP.tabela(pd.DataFrame(summary_audits), casas=12))
    print("Convenção do SD amostral conferida: ddof =", SP.pt_int(1))
    print("Os sinais negativos e as razões abaixo da referência não são truncados.")
else:
    print("PENDENTE — diferença, razão, quantis e dispersões aguardam perdas finais.")


PENDENTE — diferença, razão, quantis e dispersões aguardam perdas finais.


**O que olhar.**

- A maior divergência impressa compara o resumo armazenado ao cálculo sobre o próprio vetor.
- Diferença e razão são conferidas sem impor que a importância seja positiva.
- Coerência aritmética não comprova a origem das probabilidades que geraram as perdas.


## §3 — Reconstruir os rankings com uma implementação distinta

O executor usa o ranking do pandas. Aqui usamos `scipy.stats.rankdata` para
as posições de cada sorteio e uma ordenação explícita para as médias. Empates
entre entradas no mesmo sorteio recebem a posição média. Empates exatos entre
médias seguem a ordem das entradas declarada no protocolo.

A posição pela importância média não precisa coincidir com a média das posições.
A fração entre as primeiras posições usa o limite declarado pelo executor e o
mesmo tratamento de empates.


In [5]:
if READY:
    differences = np.asarray([row["metrics"]["logloss"]["delta"] for row in rows])
    independently_ranked = rankdata(-differences, method="average", axis=0)
    mean_order = sorted(
        range(len(rows)), key=lambda index: (-float(differences[index].mean()), index)
    )
    expected_names = [rows[index]["feature"] for index in mean_order]
    assert expected_names == RESULT["ranking"]
    rank_audits = []
    for position, index in enumerate(mean_order, start=1):
        measure = rows[index]["metrics"]["logloss"]
        current = independently_ranked[index]
        assert measure["rank"] == position and measure["ratio_rank"] == position
        np.testing.assert_array_equal(measure["rank_repeats"], current)
        np.testing.assert_allclose(
            measure["rank_mean"], current.mean(), rtol=0, atol=1e-12
        )
        np.testing.assert_allclose(
            measure["rank_sd"], current.std(ddof=1), rtol=0, atol=1e-12
        )
        np.testing.assert_allclose(
            measure["top10_fraction"], np.mean(current <= 10), rtol=0, atol=1e-12
        )
        rank_audits.append(
            {
                "posição pela média": position,
                "entrada": rows[index]["label"],
                "posição média nos sorteios": current.mean(),
                "SD da posição": current.std(ddof=1),
            }
        )
    expected_rank_sum = len(rows) * (len(rows) + 1) / 2
    np.testing.assert_allclose(
        independently_ranked.sum(axis=0), expected_rank_sum, rtol=0, atol=1e-12
    )
    print(SP.tabela(pd.DataFrame(rank_audits), casas=8))
    print("Soma das posições em cada sorteio:", SP.pt(expected_rank_sum, 1))
    print("Limite usado na fração entre as primeiras:", SP.pt_int(10))
    shown = mean_order[:10]
    fig, ax = plt.subplots(figsize=SP.SLOT)
    image = ax.imshow(
        independently_ranked[shown],
        aspect="auto",
        interpolation="nearest",
        cmap="Greys_r",
    )
    ax.set_yticks(np.arange(len(shown)), [rows[index]["label"] for index in shown])
    ax.set_xlabel("Repetição; coluna inicial representa o primeiro sorteio")
    fig.colorbar(image, ax=ax, label="Posição entre todas as entradas")
    SP.titulo(
        fig,
        "Entradas próximas podem trocar de posição entre sorteios",
        "§3 · posições conferidas independentemente",
    )
    _ = SP.salvar(fig, "course_pfi_v2_internals_ranks")
    plt.show()
else:
    print("PENDENTE — nenhum ranking foi reconstruído ou assumido como estável.")


PENDENTE — nenhum ranking foi reconstruído ou assumido como estável.


**O que olhar.**

- O mesmo vetor de perdas precisa reproduzir o ranking publicado.
- Os empates mantêm a soma das posições, sem criar entradas extras ou omissões.
- A imagem descreve a variabilidade dos sorteios, sem representar incerteza de refit.


## §4 — Conferir o vínculo dos diagnósticos de dependência

As importâncias marginais de cardiopatia e diabetes apresentadas no diagnóstico
precisam ser idênticas às do ranking. Os vetores das operações conjunta e por
subgrupos também precisam sustentar seus resumos publicados.

O agregado registra a auditoria do transporte e a quantidade de registros por
estado de diabetes. Podemos verificar consistência dessas declarações, mas não
refazer as permutações por notificação sem as fichas. Assim, não confundimos uma
invariante registrada pelo executor com uma checagem individual realizada aqui.


In [6]:
if READY:
    diagnostics = RESULT["dependence"]["results"]
    by_key = {row["key"]: row for row in diagnostics}
    assert list(by_key) == ["cardiopati", "diabetes", "joint", "within"]
    for feature in ("cardiopati", "diabetes"):
        from_ranking = next(row for row in rows if row["feature"] == feature)
        assert by_key[feature]["metrics"] == from_ranking["metrics"]
    dependence_checks = []
    for key, row in by_key.items():
        discrepancy = audit_summary(
            original, row["metrics"]["logloss"], protocol["repeats"]
        )
        dependence_checks.append(
            {
                "operação": key,
                "maior desvio absoluto": discrepancy,
                "repetições": len(row["metrics"]["logloss"]["after"]),
            }
        )
    groups = protocol["dependence"]["group_counts"]
    assert sum(groups.values()) == protocol["n_evaluation"]
    audit = RESULT["dependence"]["audit"]
    for key in (
        "joint_pair_counts_preserved",
        "within_pair_counts_preserved",
        "diabetes_unchanged_within",
    ):
        assert audit[key] is True
    assert audit["all_repeats_audited"] == protocol["repeats"]
    assert protocol["dependence"]["pair"] == ["cardiopati", "diabetes"]
    print(SP.tabela(pd.DataFrame(dependence_checks), casas=12))
    print("Quantidade total declarada nos subgrupos:", SP.pt_int(sum(groups.values())))
    print("Declarações do executor sobre o transporte: presentes e consistentes.")
    print("Transporte notificação a notificação reproduzido neste caderno:", "não")
else:
    print(
        "PENDENTE — sem diagnósticos finais, não há perdas conjuntas ou por subgrupos para conferir."
    )


PENDENTE — sem diagnósticos finais, não há perdas conjuntas ou por subgrupos para conferir.


**O que olhar.**

- As operações marginais reutilizam as mesmas perdas do ranking, sem outra seleção de casos.
- As quantidades dos subgrupos cobrem o teste completo.
- A restrição por diabetes não equivale a condicionar em todas as entradas nem a identificar causalidade.


## §5 — O que o livro diz e onde termina esta auditoria

Molnar relaciona PFI à perda de desempenho após retirar, por permutação, uma
ligação informativa de um preditor fixo. A aleatoriedade dos sorteios e a
dependência entre entradas afetam a leitura. As operações conjuntas e por
subgrupos precisam ser interpretadas segundo a informação que preservam.

Este caderno está preparado para conferir as transformações matemáticas de
um agregado, ainda ausente. A reprodução das previsões exigiria o artefato
canônico, a Gold exata e o ambiente registrado. Recalcular médias e posições
não criaria uma nova avaliação independente ou uma validação clínica.


In [7]:
if READY:
    for key in (
        "original_inputs_unchanged",
        "original_labels_unchanged",
        "original_predictions_match_frozen_evaluation",
        "canonical_model_file_unchanged",
        "source_dataset_file_unchanged",
        "source_code_unchanged",
    ):
        assert RESULT["integrity"][key] is True
    assert RESULT["integrity"]["fit_performed"] is False
    for name in (
        "srag_63_course_reference.py",
        "srag_70_explain.py",
        "srag_76_course_pfi.py",
    ):
        digest = protocol["code_sha256"][name]
        assert isinstance(digest, str) and len(digest) == 64
    print(
        "Identificadores das fontes e declarações de integridade: conferem no documento."
    )
    print("Cálculos do agregado auditados sem carregar modelo ou fichas:", "sim")
    print(
        "Probabilidades clínicas validadas independentemente por este caderno:", "não"
    )
else:
    print(
        "PENDENTE — estudo interrompido e agregado ausente; nenhuma conclusão empírica adicional aprovada."
    )
    print("Os cadernos históricos permanecem identificados como outro experimento.")


PENDENTE — estudo interrompido e agregado ausente; nenhuma conclusão empírica adicional aprovada.
Os cadernos históricos permanecem identificados como outro experimento.


**O que olhar.**

- Um hash identifica uma fonte; sua presença não certifica validade clínica.
- Coerência do resumo e validade do experimento exigem verificações distintas.
- O estado pendente mantém separados o roteiro e um resultado concluído.

## Fechamento — PENDENTE: nenhuma auditoria adicional concluída

As conferências de diferença, razão, SD e posições estão preparadas, mas
permanecem condicionadas a vetores finais que ainda não foram publicados.
O estudo completo foi interrompido antes da seleção final e sua eventual
retomada depende de decisão de escopo. A auditoria atual da aula permanece
no [internals principal](../pfi_internals.ipynb), com sua própria proveniência.


**Referências.** Molnar, *Interpretable Machine Learning*, [Permutation Feature Importance](https://christophm.github.io/interpretable-ml-book/feature-importance.html).
Scikit-learn, [log loss](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.log_loss.html).
SciPy, [rankdata](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.rankdata.html).
